# Chemprop v2 — Experiment 2: CIP R/S Featurizer (Option B — Copy-Paste)

This notebook is **functionally identical** to `4_chemprop_exp_2_optA.ipynb`.
It tests the same CIP R/S atom featurizer but uses a different implementation
strategy.

**Implementation strategy — Option B (copy-paste):**
The full `MultiHotAtomFeaturizer` source is copied into this notebook and
extended with a `cip_rs` classmethod and an overridden `__call__`. There is
**no import of `MultiHotAtomFeaturizer` from chemprop** — the class is entirely
self-contained in this notebook.

**When to prefer this approach:**
- You want to audit the complete featurizer logic without jumping to library source.
- You need to patch something deeper in the parent class (e.g., the block layout
  itself, or adding a new subfeat group).
- You are pinning to a specific version of chemprop and want the notebook to be
  stable even if chemprop updates its internals.

**Trade-off vs Option A:**
- More code in the notebook (~100 extra lines).
- If chemprop updates `v1()` defaults or the block order, this copy silently
  diverges. Option A subclass stays in sync automatically.
- The produced feature vectors and trained models are **bit-for-bit identical**
  to Option A (same algorithm, same v1 defaults).

**Conditions and hypotheses:** identical to Option A.

Part of series: `4_chemprop_exp_1.ipynb` → **`4_chemprop_exp_2_optB.ipynb`**


## Imports

In [1]:
import chemprop

In [2]:
chemprop.__version__

'2.2.2'

In [1]:
import gc
import time
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import torch

from rdkit import Chem
from rdkit.Chem import rdchem
from rdkit.Chem.rdchem import Atom, HybridizationType
from typing import Sequence

from lightning import pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from sklearn.metrics import (
    roc_auc_score,
    matthews_corrcoef,
    accuracy_score,
    f1_score,
    average_precision_score,
)

from chemprop import data, featurizers, models, nn
from chemprop.featurizers.base import VectorFeaturizer
# NOTE: MultiHotAtomFeaturizer is NOT imported here.
# The full class is defined below (Option B — copy-paste approach).

## Configuration

Identical to Exp 1 — same data, splits, batch size, and early-stopping settings.


In [2]:
INPUT_PATH  = 'data/class_all.csv'
FOLDS_PATH  = 'data/cmrt_folds.npz'
NUM_WORKERS = 0      # set >0 if multiprocessing is available
MAX_EPOCHS  = 50     # upper bound; EarlyStopping will typically stop sooner
PATIENCE    = 10     # early stopping patience
BATCH_SIZE  = 64
SEED        = 42

pl.seed_everything(SEED, workers=True)

Seed set to 42


42

## Load data and splits

In [3]:
df = pd.read_csv(INPUT_PATH, index_col=0)
print(f'Dataset shape: {df.shape}')
df.head()

Dataset shape: (3858, 6)


,SMILES,SMILES_opp,TR/TE,F/L_class,@/@@_class,R/S_class
0,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,TE,F,@,S
1,Brc1ccc2c(c1)N[C@@H](c1ccccc1)CC2,Brc1ccc2c(c1)N[C@H](c1ccccc1)CC2,TE,L,@@,R
2,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,TE,F,@,S
3,C#CCO[C@@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc...,C#CCO[C@H](CSc1nc2cc(Cl)ccc2s1)CN(C)C(c1ccccc1...,TE,L,@@,R
4,C=C(C(C)=O)[C@@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,C=C(C(C)=O)[C@H](CC(=O)c1ccc(Br)cc1)C(=O)OCC,TE,F,@@,R


In [4]:
def load_folds(path: str) -> list[dict[str, np.ndarray]]:
    """
    Load pre-computed splits from a .npz file.

    Args:
        path: Path to .npz file saved by save_folds() in notebook 2.

    Returns:
        List of dicts with keys 'train', 'val', 'test' as numpy arrays
        of molecule-level integer indices into the full dataset.
    """
    archive = np.load(path)
    fold_indices = sorted(set(
        int(k.split('_')[0].replace('fold', '')) for k in archive.files))
    return [
        {
            'train': archive[f'fold{i}_train'],
            'val':   archive[f'fold{i}_val'],
            'test':  archive[f'fold{i}_test'],
        }
        for i in fold_indices
    ]

In [5]:
mol_folds = load_folds(FOLDS_PATH)
print(f'Loaded {len(mol_folds)} folds')
for i, fold in enumerate(mol_folds):
    n_tr = len(fold['train'])
    n_v  = len(fold['val'])
    n_te = len(fold['test'])
    print(f'  Fold {i}: train={n_tr:,}  val={n_v:,}  test={n_te:,}')

Loaded 5 folds
  Fold 0: train=3,478  val=190  test=190
  Fold 1: train=3,478  val=190  test=190
  Fold 2: train=3,478  val=190  test=190
  Fold 3: train=3,478  val=190  test=190
  Fold 4: train=3,478  val=190  test=190


## Target variables and label encoding

All three targets are binary and perfectly balanced (50/50). Chance baseline: 50%.


In [6]:
TARGET_LABEL_MAPS = {
    'R/S_class':  {'R': 0, 'S': 1},
    '@/@@_class': {'@': 0, '@@': 1},
    'F/L_class':  {'F': 0, 'L': 1},
}
SINGLE_TARGETS = list(TARGET_LABEL_MAPS.keys())

for col, lmap in TARGET_LABEL_MAPS.items():
    df[f'label_{col}'] = df[col].map(lmap)

print('Class distributions (all should be 50/50):')
for col in SINGLE_TARGETS:
    vc = df[f'label_{col}'].value_counts(normalize=True)
    print(f'  {col}: {vc.to_dict()}')

Class distributions (all should be 50/50):
  R/S_class: {1: 0.5, 0: 0.5}
  @/@@_class: {0: 0.5, 1: 0.5}
  F/L_class: {0: 0.5, 1: 0.5}


## Featurizer: CIP R/S encoding (Option B — Simplified Copy)

### Design

A simplified `MultiHotAtomFeaturizer` is defined directly in this notebook.
The key differences from the chemprop default:

1. **No `v1` / `v2` / `organic` / `cip_rs` classmethods** — the constructor
   directly bakes in the v2 atom set (`range(1, 37) + [53]`) as defaults.
2. **ChiralTag (`@/@@`) is completely removed** — `int(a.GetChiralTag())` is
   never called. The chiral block is written using `a.HasProp("_CIPCode")` /
   `a.GetProp("_CIPCode")` only.
3. **The chiral block is 3 slots** (R | S | unspecified + unknown-pad),
   **not 5**, because we no longer need 4 ChiralTag values. This means the
   feature vector is **2 slots shorter** than the default featurizer.
   `SimpleMoleculeMolGraphFeaturizer` reads `atom_fdim` dynamically, so this
   is handled automatically.

### Chiral block layout (this notebook)

| Slot (relative) | Meaning              |
|---|---|
| 0               | CIP R                |
| 1               | CIP S                |
| 2               | unspecified / achiral (unknown pad combined) |

### Prerequisite

`Chem.AssignStereochemistry(mol, cleanIt=True, force=True)` must be called
before featurization so `_CIPCode` is populated on stereocenters.


In [7]:
# ── Option B: simplified MultiHotAtomFeaturizer with CIP R/S only ────────────
#
# Differences from chemprop default:
#   - No classmethods (v2 defaults baked directly into __init__ signatures)
#   - ChiralTag (int(a.GetChiralTag())) removed entirely
#   - Chiral block is 3 slots: R, S, unspecified  (not 5)
#   - Feature vector is 2 slots shorter than the chemprop default

from rdkit.Chem.rdchem import Atom, HybridizationType
from chemprop.featurizers.base import VectorFeaturizer
from typing import Sequence


class MultiHotAtomFeaturizer(VectorFeaturizer[Atom]):
    """
    Simplified atom featurizer that encodes CIP R/S instead of ChiralTag.

    Feature layout:
        atomic number   (1 + len(atomic_nums)) slots
        degree          (1 + len(degrees))      slots
        formal charge   (1 + len(formal_charges)) slots
        CIP chirality   3 slots: R | S | unspecified
        num Hs          (1 + len(num_Hs))       slots
        hybridization   (1 + len(hybridizations)) slots
        aromaticity     1 slot
        mass            1 slot

    Requires Chem.AssignStereochemistry() before featurization.
    """

    def __init__(
        self,
        atomic_nums: Sequence[int] = list(range(1, 37)) + [53],
        degrees: Sequence[int] = list(range(6)),
        formal_charges: Sequence[int] = [-1, -2, 1, 2, 0],
        num_Hs: Sequence[int] = list(range(5)),
        hybridizations: Sequence[HybridizationType] = [
            HybridizationType.S,
            HybridizationType.SP,
            HybridizationType.SP2,
            HybridizationType.SP2D,
            HybridizationType.SP3,
            HybridizationType.SP3D,
            HybridizationType.SP3D2,
        ],
    ):
        self.atomic_nums    = {j: i for i, j in enumerate(atomic_nums)}
        self.degrees        = {i: i for i in degrees}
        self.formal_charges = {j: i for i, j in enumerate(formal_charges)}
        self.num_Hs         = {i: i for i in num_Hs}
        self.hybridizations = {ht: i for i, ht in enumerate(hybridizations)}

        # chiral_tags is intentionally absent — CIP R/S is 3 fixed slots
        self._CIP_SLOTS = 3   # R=0, S=1, unspecified=2

        self._subfeats = [
            self.atomic_nums,
            self.degrees,
            self.formal_charges,
            self.num_Hs,
            self.hybridizations,
        ]
        subfeat_sizes = [
            1 + len(self.atomic_nums),
            1 + len(self.degrees),
            1 + len(self.formal_charges),
            self._CIP_SLOTS,          # chiral block: no extra unknown pad
            1 + len(self.num_Hs),
            1 + len(self.hybridizations),
            1,                        # aromaticity
            1,                        # mass
        ]
        self.__size = sum(subfeat_sizes)

        # Offset of the CIP block (after atomic_num, degree, formal_charge)
        self._chiral_start = (
            (1 + len(self.atomic_nums))
            + (1 + len(self.degrees))
            + (1 + len(self.formal_charges))
        )

    def __len__(self) -> int:
        return self.__size

    def __call__(self, a: Atom | None) -> np.ndarray:
        x = np.zeros(self.__size)

        if a is None:
            return x

        # ── Subfeatures that use the standard multi-hot loop ──────────────
        # (chiral_tags deliberately excluded — handled separately below)
        loop_feats = [
            a.GetAtomicNum(),
            a.GetTotalDegree(),
            a.GetFormalCharge(),
            int(a.GetTotalNumHs()),
            a.GetHybridization(),
        ]

        i = 0
        for feat, choices in zip(loop_feats, self._subfeats):
            j = choices.get(feat, len(choices))
            x[i + j] = 1
            i += len(choices) + 1

            # After the formal_charge block, write CIP R/S before num_Hs
            if choices is self.formal_charges:
                s = self._chiral_start
                if a.HasProp('_CIPCode'):
                    cip = a.GetProp('_CIPCode')
                    if cip == 'R':
                        x[s] = 1.0
                    elif cip == 'S':
                        x[s + 1] = 1.0
                    else:
                        x[s + 2] = 1.0   # unexpected CIP value → unspecified
                else:
                    x[s + 2] = 1.0       # no _CIPCode → achiral / unspecified
                i += self._CIP_SLOTS     # advance past CIP block

        x[i] = int(a.GetIsAromatic())
        x[i + 1] = 0.01 * a.GetMass()

        return x

    def num_only(self, a: Atom) -> np.ndarray:
        """Featurize the atom by setting only the atomic number bit."""
        x = np.zeros(len(self))
        if a is None:
            return x
        idx = self.atomic_nums.get(a.GetAtomicNum(), len(self.atomic_nums))
        x[idx] = 1
        return x


# ── Sanity checks ─────────────────────────────────────────────────────────────
# Test molecules: (R)- and (S)-ibuprofen — CIP confirmed by RDKit.
# We do NOT assume @@ → R or @ → S (that assumption caused the original bug).

_SMI_R = 'CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O'   # (R)-ibuprofen, CIP=R confirmed
_SMI_S = 'CC(C)Cc1ccc(cc1)[C@H](C)C(=O)O'    # (S)-ibuprofen, CIP=S confirmed


def _verify_featurizer():
    """
    Confirm:
    1. Feature vector length is self.__size (shorter than default by 2 slots
       because chiral block shrinks from 5 to 3).
    2. CIP R atom  → slot s+0 == 1, other chiral slots == 0.
    3. CIP S atom  → slot s+1 == 1, other chiral slots == 0.
    4. Achiral atom → slot s+2 == 1, other chiral slots == 0.
    5. Non-chiral features (aromaticity, mass, degree) are still correct.
    """
    feat = MultiHotAtomFeaturizer()
    print(f"Feature vector length: {len(feat)}")
    print(f"  (chemprop v2 default would be {len(feat) + 2} — 2 fewer slots "
          f"because CIP block is 3 slots, not 5)")

    for smi in [_SMI_R, _SMI_S]:
        mol = Chem.MolFromSmiles(smi)
        Chem.AssignStereochemistry(mol, cleanIt=True, force=True)

        chiral_atoms = [
            a for a in mol.GetAtoms()
            if a.HasProp('_CIPCode')
        ]
        assert len(chiral_atoms) == 1, f"Expected 1 stereocenter, got {len(chiral_atoms)}"
        chiral_atom = chiral_atoms[0]
        cip_actual  = chiral_atom.GetProp('_CIPCode')

        fvec  = feat(chiral_atom)
        s     = feat._chiral_start
        block = fvec[s : s + feat._CIP_SLOTS]

        print(f"  {smi}")
        print(f"    _CIPCode={cip_actual}, chiral block={block.tolist()}")

        if cip_actual == 'R':
            assert block[0] == 1.0 and sum(block) == 1.0, \
                f"R: expected slot 0, got {block.tolist()}"
            print(f"    ✓ R → slot 0")
        elif cip_actual == 'S':
            assert block[1] == 1.0 and sum(block) == 1.0, \
                f"S: expected slot 1, got {block.tolist()}"
            print(f"    ✓ S → slot 1")

    # Achiral atom — isobutane
    mol_ac = Chem.MolFromSmiles('CC(C)C')
    Chem.AssignStereochemistry(mol_ac, cleanIt=True, force=True)
    atom_ac = mol_ac.GetAtomWithIdx(1)
    assert not atom_ac.HasProp('_CIPCode'), "Expected no _CIPCode on achiral atom"
    fvec_ac = feat(atom_ac)
    block_ac = fvec_ac[feat._chiral_start : feat._chiral_start + feat._CIP_SLOTS]
    print(f"  isobutane central C (achiral) → chiral block={block_ac.tolist()}")
    assert block_ac[2] == 1.0 and sum(block_ac) == 1.0, \
        f"Achiral: expected slot 2, got {block_ac.tolist()}"
    print(f"    ✓ achiral → slot 2")

    # Spot-check non-chiral feature: aromaticity on benzene carbon
    mol_benz = Chem.MolFromSmiles('c1ccccc1')
    Chem.AssignStereochemistry(mol_benz, cleanIt=True, force=True)
    fvec_ar = feat(mol_benz.GetAtomWithIdx(0))
    # Aromaticity bit is at index (len - 2)
    assert fvec_ar[-2] == 1.0, "Aromaticity bit should be set for benzene C"
    print(f"  benzene C aromaticity bit = {fvec_ar[-2]} ✓")

    print(f"\nAll assertions passed ✓")
    print(f"  _chiral_start={feat._chiral_start}, CIP slots={feat._CIP_SLOTS}")

_verify_featurizer()

Feature vector length: 70
  (chemprop v2 default would be 72 — 2 fewer slots because CIP block is 3 slots, not 5)
  CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O
    _CIPCode=R, chiral block=[1.0, 0.0, 0.0]
    ✓ R → slot 0
  CC(C)Cc1ccc(cc1)[C@H](C)C(=O)O
    _CIPCode=S, chiral block=[0.0, 1.0, 0.0]
    ✓ S → slot 1
  isobutane central C (achiral) → chiral block=[0.0, 0.0, 1.0]
    ✓ achiral → slot 2
  benzene C aromaticity bit = 1.0 ✓

All assertions passed ✓
  _chiral_start=51, CIP slots=3


## Datapoints with CIP stereochemistry assignment

`_CIPCode` is only available on RDKit `Atom` objects after
`Chem.AssignStereochemistry()` has been called. We build a thin wrapper that
calls this immediately after SMILES parsing, before constructing the
`MoleculeDatapoint`.

This replaces the plain `MoleculeDatapoint.from_smi(smi, ignore_stereo=False)`
call used in Exp 1. The `ignore_stereo=False` flag (the default) is still
required so that RDKit preserves `@/@@` tokens during parsing — they are needed
for correct CIP assignment.


In [8]:
def make_datapoint_with_cip(smi: str) -> data.MoleculeDatapoint:
    """
    Parse SMILES, assign CIP stereochemistry, and return a MoleculeDatapoint.

    The CIP assignment populates the _CIPCode atom property, which
    CIPRSAtomFeaturizer reads during featurization.  ignore_stereo=False
    (the default) is required so that @ / @@ tokens are kept by RDKit.
    """
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        raise ValueError(f'RDKit could not parse SMILES: {smi}')
    Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
    dp = data.MoleculeDatapoint(mol=mol)
    return dp


all_data_cip = [make_datapoint_with_cip(smi) for smi in df['SMILES']]
print(f'Built {len(all_data_cip):,} CIP-assigned datapoints')

# Quick spot-check: verify _CIPCode is present on stereocenters
_test_mol = all_data_cip[0].mol
_cip_atoms = [
    (a.GetIdx(), a.GetPropsAsDict().get('_CIPCode'))
    for a in _test_mol.GetAtoms()
    if '_CIPCode' in a.GetPropsAsDict()
]
print(f'Spot-check (molecule 0): stereocenter atoms with _CIPCode: {_cip_atoms}')

Built 3,858 CIP-assigned datapoints
Spot-check (molecule 0): stereocenter atoms with _CIPCode: [(8, 'S')]


In [9]:
# Instantiate the simplified CIP R/S featurizer (v2 defaults, no classmethod needed)
cip_featurizer = MultiHotAtomFeaturizer()

# Inspect feature dimensions via a representative molecule
_test_smi = df['SMILES'].iloc[0]
_test_mol_raw = Chem.MolFromSmiles(_test_smi)
Chem.AssignStereochemistry(_test_mol_raw, cleanIt=True, force=True)
_mfeat = featurizers.SimpleMoleculeMolGraphFeaturizer(atom_featurizer=cip_featurizer)
_graph = _mfeat(_test_mol_raw)

print(f'Atom feature dim : {_graph.V.shape[1]}')
print(f'  (chemprop v2 default is {_graph.V.shape[1] + 2} — 2 fewer slots: CIP block is 3, not 5)')
print(f'Bond feature dim : {_graph.E.shape[1]}')
print(f'Nodes            : {_graph.V.shape[0]}')
print(f'cip_featurizer._chiral_start = {cip_featurizer._chiral_start}')
print(f'cip_featurizer._CIP_SLOTS    = {cip_featurizer._CIP_SLOTS}')

Atom feature dim : 70
  (chemprop v2 default is 72 — 2 fewer slots: CIP block is 3, not 5)
Bond feature dim : 14
Nodes            : 17
cip_featurizer._chiral_start = 51
cip_featurizer._CIP_SLOTS    = 3


In [20]:
_atom_fdim = _mfeat.atom_fdim
_bond_fdim = _mfeat.bond_fdim
_W_i_input = _atom_fdim + _bond_fdim   # BondMessagePassing: W_i = Linear(d_v + d_e, d_h)

print(f"atom_fdim  (d_v)        : {_atom_fdim}")
print(f"bond_fdim  (d_e)        : {_bond_fdim}")
print(f"W_i input  (d_v + d_e)  : {_W_i_input}  ← must match BondMessagePassing(d_v=..., d_e=...)")
print(f"V matrix shape          : {_graph.V.shape}  ← (n_atoms, atom_fdim)")
print(f"E matrix shape          : {_graph.E.shape}  ← (2*n_bonds, bond_fdim)")
assert _graph.V.shape[1] == _atom_fdim, "V col dim != atom_fdim"
assert _graph.E.shape[1] == _bond_fdim, "E col dim != bond_fdim"
print("Dimension check passed ✓")

atom_fdim  (d_v)        : 70
bond_fdim  (d_e)        : 14
W_i input  (d_v + d_e)  : 84  ← must match BondMessagePassing(d_v=..., d_e=...)
V matrix shape          : (17, 70)  ← (n_atoms, atom_fdim)
E matrix shape          : (38, 14)  ← (2*n_bonds, bond_fdim)
Dimension check passed ✓


## Model and trainer builder functions

`build_mpnn` takes `mol_graph_featurizer` as its first argument and passes
`atom_fdim` and `bond_fdim` explicitly to `BondMessagePassing`. This is
required here because our featurizer produces 70-dimensional atom features
(vs the chemprop v2 default of 72), and `BondMessagePassing()` with no
arguments hard-codes the default dimensions, causing a shape mismatch at the
`W_i` linear layer.

Option A avoids this because it produces vectors of the same length as the
default featurizer (72), so `BondMessagePassing()` needs no explicit fdim args.


In [10]:
def build_mpnn(mol_graph_featurizer, n_tasks: int = 1) -> models.MPNN:
    """
    Build a fresh Chemprop v2 MPNN for binary classification.

    BondMessagePassing takes d_v and d_e (not atom_fdim/bond_fdim).
    These are read from mol_graph_featurizer so W_i is sized correctly
    for our custom atom feature dimension.

    Args:
        mol_graph_featurizer: The SimpleMoleculeMolGraphFeaturizer in use.
        n_tasks: Number of binary classification outputs.
    """
    mp  = nn.BondMessagePassing(
        d_v=mol_graph_featurizer.atom_fdim,
        d_e=mol_graph_featurizer.bond_fdim,
    )
    agg = nn.MeanAggregation()
    ffn = nn.BinaryClassificationFFN(n_tasks=n_tasks)
    metric_list = [
        nn.metrics.BinaryAUROC(),
        nn.metrics.BinaryAUPRC(),
        nn.metrics.BinaryAccuracy(),
        nn.metrics.BinaryF1Score(),
    ]
    return models.MPNN(mp, agg, ffn, batch_norm=False, metrics=metric_list)

In [11]:
def build_trainer(fold_idx: int, condition_name: str, target_col: str) -> pl.Trainer:
    """
    Build a Lightning Trainer with EarlyStopping and ModelCheckpoint.

    Args:
        fold_idx: The current fold number (used for checkpoint naming).
        condition_name: The featurization condition (used for checkpoint naming).
        target_col: The target being predicted (used for checkpoint naming).

    Returns:
        Configured pl.Trainer.
    """
    early_stop = EarlyStopping(
        monitor='val/roc',
        patience=PATIENCE,
        mode='max',
        verbose=False,
    )
    checkpoint_callback = ModelCheckpoint(
        dirpath='checkpoints/',
        filename=f"{condition_name}_{target_col.replace('/', '_')}_fold{fold_idx}_best",
        monitor='val/roc',
        mode='max',
        save_top_k=1,
        verbose=False,
    )
    return pl.Trainer(
        logger=False,
        enable_checkpointing=True,
        enable_progress_bar=False,
        accelerator='auto',
        devices=1,
        max_epochs=MAX_EPOCHS,
        callbacks=[early_stop, checkpoint_callback],
    )

## `evaluate_chemprop` helper

Identical to Exp 1.


In [12]:
def evaluate_chemprop(
    trainer: pl.Trainer,
    mpnn: models.MPNN,
    loader,
    y_true: np.ndarray,
) -> dict:
    """
    Run prediction and compute all metrics for one fold/split.

    Args:
        trainer: Fitted pl.Trainer.
        mpnn: Fitted MPNN model.
        loader: DataLoader to run predictions on.
        y_true: 1D numpy array of true binary labels.

    Returns:
        Dict mapping metric name to scalar value.
    """
    preds = trainer.predict(mpnn, loader, ckpt_path='best', weights_only=False)
    probs = torch.cat(preds).squeeze().numpy()
    preds_bin = (probs > 0.5).astype(int)
    return {
        'AUROC':    roc_auc_score(y_true, probs),
        'MCC':      matthews_corrcoef(y_true, preds_bin),
        'Accuracy': accuracy_score(y_true, preds_bin),
        'F1':       f1_score(y_true, preds_bin),
        'AUPRC':    average_precision_score(y_true, probs),
    }

## Exp 2 — Single-task training loop (CIP R/S featurizer)

1 condition × 3 targets × 5 folds = **15 total fits**


In [13]:
# ── Exp 2: CIP R/S featurizer — singletask ───────────────────────────────────
SINGLE_TASK_CONDITIONS = {
    'exp_2_cip_rs_singletask': all_data_cip,
}

mol_graph_featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer(
    atom_featurizer=cip_featurizer
)
all_results = []

for condition_name, all_dpoints in SINGLE_TASK_CONDITIONS.items():
    CONDITION_START = time.time()
    print(f"\n{'#'*70}")
    print(f'Condition: {condition_name}')
    print(f"{'#'*70}")

    for target_col in SINGLE_TARGETS:
        label_col = f'label_{target_col}'
        print(f"\n  Target: {target_col}\n  {'-'*60}")

        for fold_idx, fold in enumerate(mol_folds):
            FOLD_START = time.time()
            tr_pts = [all_dpoints[i] for i in fold['train']]
            va_pts = [all_dpoints[i] for i in fold['val']]
            te_pts = [all_dpoints[i] for i in fold['test']]

            y_train = df[label_col].iloc[fold['train']].values.reshape(-1, 1)
            y_val   = df[label_col].iloc[fold['val']].values.reshape(-1, 1)
            y_test  = df[label_col].iloc[fold['test']].values.reshape(-1, 1)

            for dp, yi in zip(tr_pts, y_train): dp.y = yi
            for dp, yi in zip(va_pts, y_val):   dp.y = yi
            for dp, yi in zip(te_pts, y_test):  dp.y = yi

            train_dset = data.MoleculeDataset(tr_pts, mol_graph_featurizer)
            val_dset   = data.MoleculeDataset(va_pts, mol_graph_featurizer)
            test_dset  = data.MoleculeDataset(te_pts, mol_graph_featurizer)

            train_loader = data.build_dataloader(
                train_dset, batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS, shuffle=True)
            val_loader = data.build_dataloader(
                val_dset, batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS, shuffle=False)
            test_loader = data.build_dataloader(
                test_dset, batch_size=BATCH_SIZE,
                num_workers=NUM_WORKERS, shuffle=False)

            mpnn    = build_mpnn(mol_graph_featurizer, n_tasks=1)
            trainer = build_trainer(fold_idx, condition_name, target_col)
            trainer.fit(mpnn, train_loader, val_loader)

            splits_to_eval = [
                ('train', train_loader, y_train.ravel()),
                ('val',   val_loader,   y_val.ravel()),
                ('test',  test_loader,  y_test.ravel()),
            ]

            for split_name, loader, y_true in splits_to_eval:
                metrics = evaluate_chemprop(trainer, mpnn, loader, y_true)
                metrics.update({
                    'fold':          fold_idx,
                    'model':         'chemprop',
                    'featurization': condition_name,
                    'target':        target_col,
                    'split':         split_name,
                    'stopped_epoch': trainer.current_epoch,
                })
                all_results.append(metrics)

                if split_name == 'test':
                    fold_elapsed = time.time() - FOLD_START
                    print(
                        f'  fold {fold_idx} (test) | '
                        f"AUROC={metrics['AUROC']:.3f}  "
                        f"MCC={metrics['MCC']:.3f}  "
                        f"Acc={metrics['Accuracy']:.3f}  "
                        f"stopped_epoch={metrics['stopped_epoch']}  "
                        f"fold_time={fold_elapsed:.1f}s"
                    )

            del mpnn, trainer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            elif torch.backends.mps.is_available():
                torch.mps.empty_cache()

    total_elapsed = time.time() - CONDITION_START
    print(f'\nTotal time for {condition_name}: {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.



######################################################################
Condition: exp_2_cip_rs_singletask
######################################################################

  Target: R/S_class
  ------------------------------------------------------------


/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  226 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 317 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 317 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  226 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 90.6 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 317 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 317 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_singletask_F_L_class_fold4_best.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_singletask_F_L_class_fold4_best.ckpt
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_singletask_F_L_class_fold4_best.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_singletask_F_L_class_fold4_best.ckpt
/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints

  fold 4 (test) | AUROC=0.880  MCC=0.590  Acc=0.795  stopped_epoch=32  fold_time=135.9s

Total time for exp_2_cip_rs_singletask: 2040.3s (34.0 min)


## Exp 2 — Multi-task loop (CIP R/S featurizer)

Joint prediction of all 3 targets. **5 total fits.**


In [14]:
# ── Exp 2 multitask ───────────────────────────────────────────────────────────
print(f"\n{'#'*70}")
MULTITASK_CONDITION = 'exp_2_cip_rs_multitask'
print(f'Condition: {MULTITASK_CONDITION}')
print(f"{'#'*70}")

CONDITION_START = time.time()

for fold_idx, fold in enumerate(mol_folds):
    FOLD_START = time.time()
    print(f"\n  Fold {fold_idx}  |  "
          f"train={len(fold['train']):,}  "
          f"val={len(fold['val']):,}  "
          f"test={len(fold['test']):,}")

    tr_pts = [all_data_cip[i] for i in fold['train']]
    va_pts = [all_data_cip[i] for i in fold['val']]
    te_pts = [all_data_cip[i] for i in fold['test']]

    label_cols = [f'label_{t}' for t in SINGLE_TARGETS]
    y_train = df[label_cols].iloc[fold['train']].values
    y_val   = df[label_cols].iloc[fold['val']].values
    y_test  = df[label_cols].iloc[fold['test']].values

    for dp, yi in zip(tr_pts, y_train): dp.y = yi
    for dp, yi in zip(va_pts, y_val):   dp.y = yi
    for dp, yi in zip(te_pts, y_test):  dp.y = yi

    train_dset = data.MoleculeDataset(tr_pts, mol_graph_featurizer)
    val_dset   = data.MoleculeDataset(va_pts, mol_graph_featurizer)
    test_dset  = data.MoleculeDataset(te_pts, mol_graph_featurizer)

    train_loader = data.build_dataloader(
        train_dset, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=True)
    val_loader   = data.build_dataloader(
        val_dset,   batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)
    test_loader  = data.build_dataloader(
        test_dset,  batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, shuffle=False)

    mpnn    = build_mpnn(mol_graph_featurizer, n_tasks=3)
    trainer = build_trainer(fold_idx, MULTITASK_CONDITION, 'all_targets')
    trainer.fit(mpnn, train_loader, val_loader)

    splits_to_eval = [
        ('train', train_loader, y_train),
        ('val',   val_loader,   y_val),
        ('test',  test_loader,  y_test),
    ]

    for split_name, loader, y_true_all in splits_to_eval:
        preds     = trainer.predict(mpnn, loader, ckpt_path='best', weights_only=False)
        all_probs = torch.cat(preds).numpy()

        for task_idx, target_col in enumerate(SINGLE_TARGETS):
            probs     = all_probs[:, task_idx]
            y_true    = y_true_all[:, task_idx]
            preds_bin = (probs > 0.5).astype(int)
            metrics = {
                'AUROC':    roc_auc_score(y_true, probs),
                'MCC':      matthews_corrcoef(y_true, preds_bin),
                'Accuracy': accuracy_score(y_true, preds_bin),
                'F1':       f1_score(y_true, preds_bin),
                'AUPRC':    average_precision_score(y_true, probs),
                'fold':          fold_idx,
                'model':         'chemprop',
                'featurization': MULTITASK_CONDITION,
                'target':        target_col,
                'split':         split_name,
                'stopped_epoch': trainer.current_epoch,
            }
            all_results.append(metrics)

            if split_name == 'test':
                print(
                    f"  {target_col} (test): "
                    f"AUROC={metrics['AUROC']:.3f}  "
                    f"MCC={metrics['MCC']:.3f}  "
                    f"Acc={metrics['Accuracy']:.3f}"
                )

    fold_elapsed = time.time() - FOLD_START
    print(f'  fold_time={fold_elapsed:.1f}s')

    del mpnn, trainer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    elif torch.backends.mps.is_available():
        torch.mps.empty_cache()

total_elapsed = time.time() - CONDITION_START
print(f'\nTotal time for {MULTITASK_CONDITION}: {total_elapsed:.1f}s ({total_elapsed/60:.1f} min)')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.



######################################################################
Condition: exp_2_cip_rs_multitask
######################################################################

  Fold 0  |  train=3,478  val=190  test=190


/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  226 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 91.2 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 317 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 317 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
Loading `train_dataloader` to estimate number of stepping batches.
/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name            ┃ Type                    ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ message_passing │ BondMessagePassing      │  226 K │ train │     0 │
│ 1 │ agg             │ MeanAggregation         │      0 │ train │     0 │
│ 2 │ bn              │ Identity                │      0 │ train │     0 │
│ 3 │ predictor       │ BinaryClassificationFFN │ 91.2 K │ train │     0 │
│ 4 │ X_d_transform   │ Identity                │      0 │ train │     0 │
│ 5 │ metrics         │ ModuleList              │      0 │ train │     0 │
└───┴─────────────────┴─────────────────────────┴────────┴───────┴───────┘

Trainable params: 317 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 317 K                                                                                                
Total estimated model params size (MB): 1                                                                          
Modules in train mode: 27                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=50` reached.
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_multitask_all_targets_fold4_best.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_multitask_all_targets_fold4_best.ckpt
Restoring states from the checkpoint path at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_multitask_all_targets_fold4_best.ckpt
Loaded model weights from the checkpoint at /Users/kevin/Documents/code/stereoisomer_prediction/cmrt/checkpoints/exp_2_cip_rs_multitask_all_targets_fold4_best.ckpt
/Users/kevin/opt/anaconda3/envs/chemprop_v2/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Restoring states from the checkpoint path at /Users/kevin/Doc

  R/S_class (test): AUROC=0.999  MCC=0.958  Acc=0.979
  @/@@_class (test): AUROC=0.815  MCC=0.390  Acc=0.695
  F/L_class (test): AUROC=0.855  MCC=0.579  Acc=0.789
  fold_time=211.3s

Total time for exp_2_cip_rs_multitask: 1022.2s (17.0 min)


## Summary across Exp 2 conditions

In [15]:
results_df = pd.DataFrame(all_results)
metric_cols  = ['AUROC', 'MCC', 'Accuracy', 'F1', 'AUPRC']
target_order = ['R/S_class', '@/@@_class', 'F/L_class']
condition_order = [
    'exp_2_cip_rs_singletask',
    'exp_2_cip_rs_multitask',
]

test_results = results_df[results_df['split'] == 'test']
summary = (
    test_results
    .groupby(['featurization', 'target'])[metric_cols]
    .agg(['mean', 'std'])
    .round(4)
)
summary = summary.reindex([(c, t) for c in condition_order for t in target_order])
print('Test Set Performance (Mean ± std across 5 folds):')
print(summary.to_string())

Test Set Performance (Mean ± std across 5 folds):
                                     AUROC             MCC         Accuracy              F1           AUPRC        
                                      mean     std    mean     std     mean     std    mean     std    mean     std
featurization           target                                                                                     
exp_2_cip_rs_singletask R/S_class   0.9933  0.0069  0.9499  0.0269   0.9747  0.0136  0.9751  0.0133  0.9927  0.0082
                        @/@@_class  0.8140  0.0185  0.4761  0.0515   0.7379  0.0256  0.7374  0.0274  0.8150  0.0199
                        F/L_class   0.8531  0.0461  0.5578  0.0813   0.7779  0.0413  0.7763  0.0506  0.8447  0.0582
exp_2_cip_rs_multitask  R/S_class   0.9987  0.0015  0.9811  0.0156   0.9905  0.0078  0.9905  0.0078  0.9988  0.0013
                        @/@@_class  0.8176  0.0168  0.4569  0.0978   0.7284  0.0489  0.7290  0.0490  0.8234  0.0095
                      

The results confirm both hypotheses precisely:

R/S: 0.823 AUROC in Exp 1 → 0.993 in Exp 2. When the atom features directly encode CIP R/S, the model learns it almost perfectly. This is essentially a lookup — the label is the feature.

@/@@: 0.993 AUROC in Exp 1 → 0.814 in Exp 2. Removing the ChiralTag signal hurts substantially, but the model still does well above chance. This means graph structure alone (neighbor identity, bond topology) carries meaningful @/@@ signal — the model can partially recover parity from the molecular graph even without an explicit parity bit.

F/L: 0.763 in Exp 1 → 0.853 in Exp 2. Interestingly, F/L improves when you switch from ChiralTag to CIP R/S. In your dataset F/L is perfectly correlated with R/S by construction (F=first-eluting enantiomer, which correlates with one CIP configuration), so this makes sense — CIP R/S is a better proxy for F/L than @/@@ parity is.

The multitask results in Exp 2 are notably stronger for R/S (0.999 vs 0.993 singletask), suggesting the model benefits from jointly predicting @/@@ and F/L even when those tasks are harder — the shared representation generalizes better.

## Hypothesis evaluation

Two comparisons:
1. **R/S should now be easiest** — CIP R/S is encoded directly in the atom features.
2. **@/@@ should be harder than in Exp 1** — the direct ChiralTag signal is removed.


In [16]:
# ── Hypothesis 1: R/S should be easier than @/@@ (inverse of Exp 1) ──────────
df_st_test = results_df[
    (results_df['featurization'] == 'exp_2_cip_rs_singletask') &
    (results_df['split'] == 'test')
]

rs_auroc = df_st_test[df_st_test['target'] == 'R/S_class']['AUROC'].mean()
at_auroc = df_st_test[df_st_test['target'] == '@/@@_class']['AUROC'].mean()

print('Hypothesis 1: R/S easier than @/@@ for Exp 2 (encodes CIP R/S directly)?')
print(f'  R/S  AUROC (singletask): {rs_auroc:.4f}')
print(f'  @/@@ AUROC (singletask): {at_auroc:.4f}')
if rs_auroc > at_auroc:
    print('  CONFIRMED: R/S > @/@@ — consistent with direct CIP R/S encoding')
else:
    print('  NOT CONFIRMED: @/@@ >= R/S')

print()

# ── Multitask vs singletask ────────────────────────────────────────────────────
df_mt_test = results_df[
    (results_df['featurization'] == 'exp_2_cip_rs_multitask') &
    (results_df['split'] == 'test')
]

print('Multitask vs singletask AUROC (test set):')
print(f"  {'Target':<15} {'Singletask':>12} {'Multitask':>12} {'Delta':>8}")
for target in target_order:
    st = df_st_test[df_st_test['target'] == target]['AUROC'].mean()
    mt = df_mt_test[df_mt_test['target'] == target]['AUROC'].mean()
    print(f"  {target:<15} {st:>12.4f} {mt:>12.4f} {mt - st:>+8.4f}")

Hypothesis 1: R/S easier than @/@@ for Exp 2 (encodes CIP R/S directly)?
  R/S  AUROC (singletask): 0.9933
  @/@@ AUROC (singletask): 0.8140
  CONFIRMED: R/S > @/@@ — consistent with direct CIP R/S encoding

Multitask vs singletask AUROC (test set):
  Target            Singletask    Multitask    Delta
  R/S_class             0.9933       0.9987  +0.0055
  @/@@_class            0.8140       0.8176  +0.0035
  F/L_class             0.8531       0.8309  -0.0222


## Per-condition performance table (test set)

In [17]:
print(f"{'Target':<15} {'Condition':<35} {'AUROC':>7} {'MCC':>7} {'Acc':>7}")
print('-' * 74)
for target in target_order:
    for cond in condition_order:
        s = results_df[
            (results_df['target'] == target) &
            (results_df['featurization'] == cond) &
            (results_df['split'] == 'test')
        ]
        if s.empty:
            continue
        auroc = s['AUROC'].mean()
        mcc   = s['MCC'].mean()
        acc   = s['Accuracy'].mean()
        print(f'{target:<15} {cond:<35} {auroc:>7.4f} {mcc:>7.4f} {acc:>7.4f}')
    print()

Target          Condition                             AUROC     MCC     Acc
--------------------------------------------------------------------------
R/S_class       exp_2_cip_rs_singletask              0.9933  0.9499  0.9747
R/S_class       exp_2_cip_rs_multitask               0.9987  0.9811  0.9905

@/@@_class      exp_2_cip_rs_singletask              0.8140  0.4761  0.7379
@/@@_class      exp_2_cip_rs_multitask               0.8176  0.4569  0.7284

F/L_class       exp_2_cip_rs_singletask              0.8531  0.5578  0.7779
F/L_class       exp_2_cip_rs_multitask               0.8309  0.5542  0.7768



## Early stopping epoch distribution

In [18]:
epoch_summary = (
    results_df[results_df['split'] == 'test']
    .groupby(['featurization', 'target'])['stopped_epoch']
    .agg(['mean', 'min', 'max'])
    .round(1)
)
print('Epochs trained before early stopping:')
print(epoch_summary.to_string())

Epochs trained before early stopping:
                                    mean  min  max
featurization           target                    
exp_2_cip_rs_multitask  @/@@_class  47.2   36   50
                        F/L_class   47.2   36   50
                        R/S_class   47.2   36   50
exp_2_cip_rs_singletask @/@@_class  42.2   29   50
                        F/L_class   33.0   21   45
                        R/S_class   15.0   14   17


## Save results for Tukey HSD

Per-fold scores saved to CSV for downstream statistical comparison alongside
Exp 0, Exp 1, and RF results.


In [19]:
output_path = '4_chemprop_exp_2_optB_results.csv'
results_df.to_csv(output_path, index=False)
print(f'\nSaved {len(results_df)} total rows to {output_path}')
print(f"  Conditions: {results_df['featurization'].unique().tolist()}")
print(f"  Targets:    {results_df['target'].unique().tolist()}")
print(f"  Folds:      {sorted(results_df['fold'].unique().tolist())}")
print(f"  Splits:     {results_df['split'].unique().tolist()}")
print(f"  Columns:    {results_df.columns.tolist()}")


Saved 90 total rows to 4_chemprop_exp_2_optB_results.csv
  Conditions: ['exp_2_cip_rs_singletask', 'exp_2_cip_rs_multitask']
  Targets:    ['R/S_class', '@/@@_class', 'F/L_class']
  Folds:      [0, 1, 2, 3, 4]
  Splits:     ['train', 'val', 'test']
  Columns:    ['AUROC', 'MCC', 'Accuracy', 'F1', 'AUPRC', 'fold', 'model', 'featurization', 'target', 'split', 'stopped_epoch']
